ESERCIZIO

Implementa una funzione che utilizzi PyMuPDF per estrarre solo la prima pagina di un CV e utilizzi una semplice espressione regolare per cercare entitià email
Successivamente, scrivi uno script che calcoli la similarità coseno tra due frasi predefinite utilizzanod numpy

In [1]:
import os
import re
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import fitz  # PyMuPDF


def estrai_email_da_cv(nome_file):
    """
    Estrae il testo dalla prima pagina di un PDF e cerca una email tramite Regex.
    
    Args:
        nome_file (str): Nome del file PDF nella cartella dello script.
    Returns:
        tuple: (testo_estratto, email_trovata)
    """
    # Risoluzione del percorso assoluto nella directory dello script
    diretorio_corrente = os.path.dirname(os.path.abspath(__file__))
    percorso_completo = os.path.join(diretorio_corrente, nome_file)
    
    testo_prima_pagina = ""
    email = "Nessuna email trovata"

    try:
        # 1. Apertura del documento
        with fitz.open(percorso_completo) as doc:
            # Carichiamo solo la prima pagina (indice 0)
            pagina = doc.load_page(0)
            testo_prima_pagina = pagina.get_text()

        # 2. Definizione Regex per Email
        # Spiegazione: cerca caratteri alfanumerici + @ + dominio
        pattern_email = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
        match = re.search(pattern_email, testo_prima_pagina)
        
        if match:
            email = match.group(0)

    except Exception as e:
        testo_prima_pagina = f"Errore: {e}"

    return testo_prima_pagina, email


def calcola_similarita_numpy(vettore1, vettore2):
    """
    Calcola la similarità coseno tra due vettori utilizzando esclusivamente NumPy.
    
    Args:
        vettore1 (np.array): Primo vettore.
        vettore2 (np.array): Secondo vettore.
    Returns:
        float: Valore di similarità tra 0 e 1.
    """
    # Formula: (A dot B) / (||A|| * ||B||)
    prodotto_scalare = np.dot(vettore1, vettore2)
    norma1 = np.linalg.norm(vettore1)
    norma2 = np.linalg.norm(vettore2)
    
    return prodotto_scalare / (norma1 * norma2)


# --- ESECUZIONE DELLO SCRIPT ---

if __name__ == "__main__":
    print("--- PARTE 1: Estrazione Email ---")
    file_target = "cv_test.pdf" # Assicurati che il file sia nella stessa cartella
    testo, email_estratta = estrai_email_da_cv(file_target)
    
    print(f"File analizzato: {file_target}")
    print(f"Email rilevata: {email_estratta}")

    print("\n--- PARTE 2: Similarità Semantica (NumPy) ---")
    
    # Frasi di esempio
    frase_cv = "Esperto sviluppatore Python con competenze in Deep Learning."
    frase_jd = "Ricerca programmatore Python specializzato in Intelligenza Artificiale."

    # Inizializziamo un modello Transformer leggero (Pure PyTorch) per ottenere i vettori
    # Usiamo 'bert-base-multilingual-cased' per supportare l'italiano
    tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
    modello = AutoModel.from_pretrained("bert-base-multilingual-cased")

    # Trasformiamo le frasi in vettori (Embedding)
    with torch.no_grad():
        # Tokenizzazione
        tokens_cv = tokenizer(frase_cv, return_tensors="pt", padding=True, truncation=True)
        tokens_jd = tokenizer(frase_jd, return_tensors="pt", padding=True, truncation=True)
        
        # Estrazione output (prendiamo il pooled_output e convertiamo in NumPy)
        vec_cv = modello(**tokens_cv).pooler_output[0].numpy()
        vec_jd = modello(**tokens_jd).pooler_output[0].numpy()

    # Calcolo del punteggio usando la nostra funzione NumPy
    score = calcola_similarita_numpy(vec_cv, vec_jd)

    print(f"Frase 1: {frase_cv}")
    print(f"Frase 2: {frase_jd}")
    print(f"Punteggio di similarità: {score:.4f}")

c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- PARTE 1: Estrazione Email ---


NameError: name '__file__' is not defined